# Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                              r2_score, accuracy_score, confusion_matrix,
                              classification_report, roc_curve, auc)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from scipy import stats

import warnings
warnings.filterwarnings('ignore')

# Data Loading

In [2]:
# Primary dataset: NT Government Region Populations 1986–2025
pop = pd.read_excel('nt-government-regions_1986-to-2025.xlsx')

# Secondary datasets: NT Crime Statistics
crime_nov = pd.read_csv('nt_crime_statistics_nov_2023.csv')   # 2008–2023
crime_dec = pd.read_csv('nt_crime_statistics_dec_2025.csv')   # 2023–2025

# Standardise column names to snake_case
for df in [crime_nov, crime_dec, pop]:
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]

# Preprocessing & Data Integration

## Split crime datasets by time to avoid duplication

In [3]:
# Nov 2023 file: 2008–2023; Dec 2025 file: 2023–2025
# Use Nov for 2008–2022, Dec for 2023–2025
crime_nov_hist = (crime_nov[crime_nov['year'] <= 2022]
                  .copy()
                  .rename(columns={'reporting_region': 'region'}))
crime_dec_new  = (crime_dec[crime_dec['reporting_region'] != 'Unknown']
                  .copy()
                  .rename(columns={'reporting_region': 'region',
                                   'offence_type_': 'offence_type'}))

## Region Harminisation

In [4]:
# Crime data uses 7 reporting regions; population data uses 6 NTG service regions.
# Mapping based on geographic alignment (see Assessment 1, p.3).
# This is what we termed "region harmonisation" in our project plan.
region_map = {
    'Alice Springs':  'Central Australia',
    'Darwin':         'Greater Darwin',
    'Palmerston':     'Greater Darwin',   # Palmerston is part of Greater Darwin
    'Katherine':      'Big Rivers',
    'Tennant Creek':  'Barkly',
    'Nhulunbuy':      'East Arnhem',
    'NT Balance':     'Top End'
}
for df in [crime_nov_hist, crime_dec_new]:
    df['pop_region'] = df['region'].map(region_map)

### Aggregate crime to region-year-month-category level

In [5]:
def aggregate_crime(df):
    """Aggregate offences by year, month, region, and offence category."""
    return (df.groupby(['year', 'month_number', 'pop_region', 'region', 'offence_category'])
             .agg(total_offences=('number_of_offences', 'sum'))
             .reset_index())

crime_all = (pd.concat([aggregate_crime(crime_nov_hist),
                        aggregate_crime(crime_dec_new)],
                       ignore_index=True)
               .drop_duplicates())

### Total offences per region-year-month

In [6]:
crime_rym = (crime_all
             .groupby(['year', 'month_number', 'pop_region', 'region'])
             .agg(total_offences=('total_offences', 'sum'))
             .reset_index())

## Compute alcohol and DV involvement flags

In [7]:
# Note: alcohol/DV flags are binary Yes/No — count rows flagged as 'Yes'
def get_involvement_flags(df):
    """Extract alcohol and domestic violence involvement counts."""
    df2 = df.copy()
    df2['alc'] = (df2['alcohol_involvement'].str.lower()
                  .str.contains('yes').fillna(False).astype(int))
    df2['dv']  = (df2['dv_involvement'].str.lower()
                  .str.contains('yes').fillna(False).astype(int))
    return (df2.groupby(['year', 'month_number', 'pop_region', 'region'])
               .agg(alc_offences=('alc', 'sum'),
                    dv_offences=('dv', 'sum'))
               .reset_index())

flags_all = (pd.concat([get_involvement_flags(crime_nov_hist),
                        get_involvement_flags(crime_dec_new)])
               .drop_duplicates())

crime_rym = crime_rym.merge(flags_all,
                             on=['year', 'month_number', 'pop_region', 'region'],
                             how='left')

## Aggregate population to pop_region-year level

In [8]:
pop_total = (pop.groupby(['year', 'region'])['population']
               .sum()
               .reset_index()
               .rename(columns={'region': 'pop_region'}))

## Merge crime + population

In [9]:
merged = (crime_rym
          .merge(pop_total, on=['year', 'pop_region'], how='left')
          .dropna(subset=['population']))

## Feature Engineering

In [10]:
# Offence rate normalised per 1,000 residents (standard criminology metric)
merged['offence_rate_per_1000'] = (merged['total_offences'] / merged['population']) * 1000
# Proportional involvement rates (0–1)
merged['alc_rate'] = merged['alc_offences'] / merged['total_offences'].replace(0, np.nan)
merged['dv_rate']  = merged['dv_offences']  / merged['total_offences'].replace(0, np.nan)
# Log-transform population to reduce skewness (justified: population range is ~5k–250k)
merged['log_population'] = np.log(merged['population'])
# Year index for temporal trend modelling
merged['year_index'] = merged['year'] - merged['year'].min()
merged = merged.fillna(0)

## Add demographic features from population dataset

In [11]:
pop_demo = (pop.groupby(['year', 'region'])
               .apply(lambda g: pd.Series({
                   'total_pop': g['population'].sum(),
                   'aboriginal_pop': g.loc[
                       g['aboriginal_status'].str.contains('Aboriginal', na=False),
                       'population'].sum()
               }))
               .reset_index()
               .rename(columns={'region': 'pop_region'}))
pop_demo['aboriginal_pct'] = (pop_demo['aboriginal_pop'] /
                               pop_demo['total_pop'] * 100)

merged2 = (merged
           .merge(pop_demo[['year', 'pop_region', 'aboriginal_pct']],
                  on=['year', 'pop_region'], how='left')
           .fillna(0))

print(f"Final merged dataset: {merged.shape[0]} rows × {merged.shape[1]} cols")
print(f"Year range: {merged['year'].min()}–{merged['year'].max()}")
print(f"Regions: {sorted(merged['region'].unique())}")
print(f"Null values:\n{merged.isnull().sum()}")

Final merged dataset: 1435 rows × 13 cols
Year range: 2008–2025
Regions: ['Alice Springs', 'Darwin', 'Katherine', 'NT Balance', 'Nhulunbuy', 'Palmerston', 'Tennant Creek']
Null values:
year                     0
month_number             0
pop_region               0
region                   0
total_offences           0
alc_offences             0
dv_offences              0
population               0
offence_rate_per_1000    0
alc_rate                 0
dv_rate                  0
log_population           0
year_index               0
dtype: int64


# RQ 1 - Temporal Analysis

In [12]:
# RQ1: How have total recorded offences changed over time across the NT, and which offence categories show the strongest temporal variation?

yearly       = crime_all.groupby(['year', 'offence_category'])['total_offences'].sum().reset_index()
yearly_total = crime_all.groupby('year')['total_offences'].sum().reset_index()

# For reporting: temporal variation (coefficient of variation) per category
cv_by_cat = (yearly.groupby('offence_category')['total_offences']
             .agg(['mean', 'std'])
             .assign(cv=lambda d: d['std'] / d['mean'])
             .sort_values('cv', ascending=False))
print("\nRQ1 — Coefficient of Variation by Offence Category:")
print(cv_by_cat.round(3).to_string())


RQ1 — Coefficient of Variation by Offence Category:
                                                                  mean       std     cv
offence_category                                                                       
01 Homicide                                                     20.333    19.009  0.935
04 Harm or endanger persons                                    760.000   599.253  0.788
05 Robbery, blackmail, and extortion                           286.000   219.984  0.769
061 Burglary - dwelling                                       1861.667  1431.881  0.769
02 Assault                                                    7480.333  5694.653  0.761
03 Sexual offences                                             482.000   366.337  0.760
07 Theft                                                      5605.333  4259.390  0.760
11 Property damage offences                                   5585.333  4227.013  0.757
062 Burglary - non-residential                                1050.

# RQ 2 - Explanatory Analysis

In [13]:
# RQ2: To what extent do demographic indicators explain variation in recorded offence rates across regions?

rq2 = (merged2.groupby(['year', 'pop_region', 'region'])
       .agg(total_offences=('total_offences', 'sum'),
            total_population=('population', 'first'),
            aboriginal_pct=('aboriginal_pct', 'first'),
            alc_rate=('alc_rate', 'mean'),
            dv_rate=('dv_rate', 'mean'))
       .reset_index())
rq2['offence_rate'] = rq2['total_offences'] / rq2['total_population'] * 1000
rq2['log_pop']      = np.log(rq2['total_population'])
rq2 = rq2.dropna()

features_rq2 = ['log_pop', 'aboriginal_pct', 'alc_rate', 'dv_rate', 'year']
X2 = rq2[features_rq2].values
y2 = rq2['offence_rate'].values

scaler2 = StandardScaler()
X2s = scaler2.fit_transform(X2)
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2s, y2, test_size=0.2, random_state=42)

mlr = LinearRegression()
mlr.fit(X2_train, y2_train)
y2_pred = mlr.predict(X2_test)

mae2  = mean_absolute_error(y2_test, y2_pred)
rmse2 = np.sqrt(mean_squared_error(y2_test, y2_pred))
r2_2  = r2_score(y2_test, y2_pred)

cv_r2  = cross_val_score(LinearRegression(), X2s, y2, cv=5, scoring='r2')
cv_mae = cross_val_score(LinearRegression(), X2s, y2, cv=5,
                          scoring='neg_mean_absolute_error')

# Statistical test: t-test on residuals (H0: mean residual = 0)
residuals2 = y2_test - y2_pred
t_stat2, p_val2 = stats.ttest_1samp(residuals2, 0)

print("\n=== RQ2: Multiple Linear Regression Results ===")
print(f"  MAE  = {mae2:.3f} offences/1,000")
print(f"  RMSE = {rmse2:.3f} offences/1,000")
print(f"  R²   = {r2_2:.3f}")
print(f"  5-Fold CV R²  = {cv_r2.mean():.3f} ± {cv_r2.std():.3f}")
print(f"  5-Fold CV MAE = {-cv_mae.mean():.3f} ± {cv_mae.std():.3f}")
print(f"  T-test residuals: t={t_stat2:.4f}, p={p_val2:.4f}")
print(f"  Coefficients:")
for feat, coef in zip(features_rq2, mlr.coef_):
    print(f"    {feat:20s}: {coef:+.4f}")


=== RQ2: Multiple Linear Regression Results ===
  MAE  = 56.300 offences/1,000
  RMSE = 80.463 offences/1,000
  R²   = 0.643
  5-Fold CV R²  = 0.539 ± 0.175
  5-Fold CV MAE = 59.145 ± 13.823
  T-test residuals: t=0.9850, p=0.3341
  Coefficients:
    log_pop             : -42.7567
    aboriginal_pct      : -0.0000
    alc_rate            : -205.8586
    dv_rate             : +201.9530
    year                : +2.3397


# RQ 3 - Predictive Regression

In [14]:
# RQ3: How accurately can monthly offence counts be predicted from recent crime history, offence composition, and population demographics?

rq3 = merged.sort_values(['region', 'year', 'month_number']).copy()
rq3['prev_month_offences'] = rq3.groupby('region')['total_offences'].shift(1)
rq3['prev_3mo_avg'] = (rq3.groupby('region')['total_offences']
                         .transform(lambda x: x.shift(1).rolling(3).mean()))
rq3 = rq3.dropna()

features_rq3 = ['prev_month_offences', 'prev_3mo_avg', 'log_population',
                 'alc_rate', 'dv_rate', 'month_number', 'year_index']
X3 = rq3[features_rq3].values
y3 = rq3['total_offences'].values

# Time-based split (shuffle=False preserves temporal order)
X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3, test_size=0.2, random_state=42, shuffle=False)
scaler3 = StandardScaler()
X3s = scaler3.fit_transform(X3_train)
X3ts = scaler3.transform(X3_test)

# Model 1: OLS
lr3 = LinearRegression()
lr3.fit(X3s, y3_train)
y3_pred = lr3.predict(X3ts)

# Model 2: Ridge
ridge = Ridge(alpha=1.0)
ridge.fit(X3s, y3_train)
y3_pred_ridge = ridge.predict(X3ts)

mae3_lr   = mean_absolute_error(y3_test, y3_pred)
rmse3_lr  = np.sqrt(mean_squared_error(y3_test, y3_pred))
r2_3_lr   = r2_score(y3_test, y3_pred)
mae3_r    = mean_absolute_error(y3_test, y3_pred_ridge)
rmse3_r   = np.sqrt(mean_squared_error(y3_test, y3_pred_ridge))
r2_3_r    = r2_score(y3_test, y3_pred_ridge)

# Welch's t-test: do OLS and Ridge produce significantly different errors?
# H0: mean(|residuals_OLS|) = mean(|residuals_Ridge|)
res_lr    = np.abs(y3_test - y3_pred)
res_ridge = np.abs(y3_test - y3_pred_ridge)
t_comp, p_comp = stats.ttest_ind(res_lr, res_ridge, equal_var=False)

print("\n=== RQ3: Predictive Regression Results ===")
print(f"  OLS:   MAE={mae3_lr:.1f}  RMSE={rmse3_lr:.1f}  R²={r2_3_lr:.3f}")
print(f"  Ridge: MAE={mae3_r:.1f}  RMSE={rmse3_r:.1f}  R²={r2_3_r:.3f}")
print(f"  Welch t-test OLS vs Ridge: t={t_comp:.4f}, p={p_comp:.4f}")
print(f"  → p={p_comp:.3f} > 0.05: no significant difference in error; "
      f"OLS preferred for interpretability")


=== RQ3: Predictive Regression Results ===
  OLS:   MAE=27.8  RMSE=37.6  R²=0.850
  Ridge: MAE=27.8  RMSE=37.6  R²=0.850
  Welch t-test OLS vs Ridge: t=-0.0063, p=0.9950
  → p=0.995 > 0.05: no significant difference in error; OLS preferred for interpretability


# RQ 4 - Classification

In [15]:
# RQ4: Can region-month observations be classified into higher-risk and lower-risk offence periods using crime and demographic features?

threshold = merged['offence_rate_per_1000'].quantile(0.75)
merged['high_risk'] = (merged['offence_rate_per_1000'] >= threshold).astype(int)
print(f"\nRQ4 Risk Threshold (75th pctile): {threshold:.2f} offences/1,000")
print(f"Class distribution: {merged['high_risk'].value_counts().to_dict()}")

features_rq4 = ['log_population', 'alc_rate', 'dv_rate',
                 'month_number', 'year_index', 'total_offences']
X4 = merged[features_rq4].values
y4 = merged['high_risk'].values

X4_train, X4_test, y4_train, y4_test = train_test_split(
    X4, y4, test_size=0.2, random_state=42)
scaler4 = StandardScaler()
X4s  = scaler4.fit_transform(X4_train)
X4ts = scaler4.transform(X4_test)

# Logistic Regression
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X4s, y4_train)
y4_pred_lr = log_reg.predict(X4ts)

# Decision Tree
dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=10, random_state=42)
dt.fit(X4s, y4_train)
y4_pred_dt = dt.predict(X4ts)

print("\n=== RQ4: Logistic Regression ===")
print(f"  Accuracy: {accuracy_score(y4_test, y4_pred_lr):.3f}")
print(classification_report(y4_test, y4_pred_lr,
                             target_names=['Low Risk', 'High Risk']))

print("=== RQ4: Decision Tree ===")
print(f"  Accuracy: {accuracy_score(y4_test, y4_pred_dt):.3f}")
print(classification_report(y4_test, y4_pred_dt,
                             target_names=['Low Risk', 'High Risk']))

# McNemar Test: H0 — both classifiers make the same errors
correct_lr = (y4_pred_lr == y4_test)
correct_dt = (y4_pred_dt == y4_test)
b = np.sum(correct_lr & ~correct_dt)
c = np.sum(~correct_lr & correct_dt)
if b + c > 0:
    chi2_mc = (abs(b - c) - 1)**2 / (b + c)
    from scipy.stats import chi2 as chi2_dist
    p_mcnemar = 1 - chi2_dist.cdf(chi2_mc, df=1)
    print(f"McNemar Test (LR vs DT): chi²={chi2_mc:.4f}, p={p_mcnemar:.4f}")
    if p_mcnemar < 0.05:
        print("  → Classifiers differ significantly (p < 0.05)")
    else:
        print("  → No significant difference (p ≥ 0.05); DT preferred for interpretability")


RQ4 Risk Threshold (75th pctile): 14.33 offences/1,000
Class distribution: {0: 1076, 1: 359}

=== RQ4: Logistic Regression ===
  Accuracy: 0.944
              precision    recall  f1-score   support

    Low Risk       0.95      0.98      0.96       220
   High Risk       0.93      0.82      0.87        67

    accuracy                           0.94       287
   macro avg       0.94      0.90      0.92       287
weighted avg       0.94      0.94      0.94       287

=== RQ4: Decision Tree ===
  Accuracy: 0.972
              precision    recall  f1-score   support

    Low Risk       0.97      1.00      0.98       220
   High Risk       0.98      0.90      0.94        67

    accuracy                           0.97       287
   macro avg       0.98      0.95      0.96       287
weighted avg       0.97      0.97      0.97       287

McNemar Test (LR vs DT): chi²=3.0625, p=0.0801
  → No significant difference (p ≥ 0.05); DT preferred for interpretability


# Visualisation

In [16]:
# See figures: fig_rq1_temporal_analysis.png, fig_rq2_regression.png, fig_rq3_predictive.png, fig_rq4_classification.png, fig_summary_dashboard.png

print("\n✓ All analyses complete. See output PNG files for visualisations.")


✓ All analyses complete. See output PNG files for visualisations.
